# Momentum Screener Metrics Demo
This notebook demonstrates the use of the `momentum_metrics` module to compute robust, cross-ticker comparable metrics for macro/sector/factor rotation.

In [1]:
from momentum_metrics import fetch_prices, compute_metrics
import pandas as pd

In [2]:
import json

def concat_lists_from_json(json_path, list_names):
    """
    Given a JSON file containing a dict of lists, and a list of keys (list_names),
    return a single concatenated list of all lists corresponding to those keys.
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
    result = []
    for name in list_names:
        result.extend(data.get(name, []))
    return result
json_path = 'C:\\Users\\wongb\\compounding-focus-finance-tooling\\compounding-focus-finance-tooling\\holdings.json'
etf_list = ['trumprx', 'discount_store']
etf_tickers = concat_lists_from_json(json_path, etf_list)

In [6]:
held_tickers = ['VOO', 'COPX', 'GDX', 'IAU', 'XAR', 'SIVR', 'SIL', 'AGQ', 'SHNY', 'INTC', 'AMD', 'SGI', 'GS', 'JNJ', 'HSBC', 'GLW', 'SIVR', 'EPOL', 'EWU', 'AZN', 'MRK', 'VTV', 'SCHD']
test_tickers = None
tickers =  ["VOO", 'SCHD', 'PKB', 'DTCR', 'SOXX', 'XLP', 'XLV', 'MADE']
end = pd.Timestamp.today().strftime('%Y-%m-%d')
start = (pd.Timestamp.today() - pd.Timedelta(days=3*365)).strftime('%Y-%m-%d')
prices = fetch_prices(tickers, start, end)
metrics = compute_metrics(prices, benchmark='VOO')

# Add a column to flag uptrend (1 = in uptrend, 0 = not in uptrend)
metrics['Uptrend (Ratio>SMA200)'] = metrics['ratio_above_sma200_binary']

# Sort by relative strength, but keep all tickers for comparison
metrics_sorted = metrics.sort_values('rs_3m', ascending=False)

# Print tickers that failed the initial screen (not in uptrend)
failed_tickers = set(metrics.index[metrics['ratio_above_sma200_binary'] == 0])
if failed_tickers:
    print('Tickers failing the screen (not in uptrend):', ', '.join(sorted(failed_tickers)))
else:
    print('All tickers passed the uptrend screen.')

# Select columns to display and rename for readability
cols = {
    'rs_12m_ex1': '12M Rel Strength Ex 1M',
    'rs_6m_ex1': '6M Rel Strength Ex 1M',
    'rs_slope_6m_ex1': '6M Rel Slope',
    'rs_3m': '3M Rel Strength',
    'rs_slope_3m': '3M Rel Slope',
    "pe_ratio": "PE Ratio",
    "fcf_margin": "FCF Margin",
    "roic": "ROIC",
    'distribution_days_30': 'Distribution Days (30d)',
    'distribution_days_10': 'Distribution Days (10d)',
    'uvp_pct_40d': 'UVP (%) (40d)',
    'uvp_pct_10d': 'UVP (%) (10d)',
    'uvp_slope_40d': 'UVP Slope (40d, 1000x)',
    'mom_eff_6m_ex1': 'Momentum Efficiency (6M)',
    "mom_eff_slope_6m_ex1": 'Momentum Efficiency Slope (6M, 100x)',
    'ratio_above_sma200': 'Ratio/SMA200',
    'ratio_sma200_slope_6m': 'SMA200 Slope (6M, 100x)',
    'pct_days_ratio_above_sma200_6m': '%Days Ratio>SMA200 (6M)',
    'rs_vol_3m': 'RS Volatility (3M)',
    "rs_vol_slope_3m": 'RS Volatility Slope (3M, 10000x)',
    'rs_max_dd_6m_ex1': 'RS Max Drawdown (6M)',
}

# Rename columns for display
metrics_disp = metrics_sorted[list(cols.keys())].rename(columns=cols)

# Use pandas built-in background_gradient for green and format to 2 decimals (no trailing zeros)
styled = metrics_disp.style.format('{:.2f}').background_gradient(cmap='Greens')
styled

All tickers passed the uptrend screen.


C:\Users\wongb\AppData\Roaming\Python\Python313\site-packages\pandas\io\formats\style.py:4202: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
C:\Users\wongb\AppData\Roaming\Python\Python313\site-packages\pandas\io\formats\style.py:4203: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax


,12M Rel Strength Ex 1M,6M Rel Strength Ex 1M,6M Rel Slope,3M Rel Strength,3M Rel Slope,PE Ratio,FCF Margin,ROIC,Distribution Days (30d),Distribution Days (10d),UVP (%) (40d),UVP (%) (10d),"UVP Slope (40d, 1000x)",Momentum Efficiency (6M),"Momentum Efficiency Slope (6M, 100x)",Ratio/SMA200,"SMA200 Slope (6M, 100x)",%Days Ratio>SMA200 (6M),RS Volatility (3M),"RS Volatility Slope (3M, 10000x)",RS Max Drawdown (6M)
ticker,,,,,,,,,,,,,,,,,,,,,
DTCR,19.83,10.32,0.05,18.91,0.33,17.51,nan,nan,8.00,3.00,0.71,0.65,0.14,0.10,-0.25,1.23,0.03,76.19,19.53,2.41,-10.22
SOXX,24.53,19.77,0.17,16.09,0.34,42.99,nan,nan,7.00,3.00,0.58,0.47,0.39,0.14,-0.03,1.25,0.06,100.00,26.94,0.70,-8.96
SCHD,-8.40,-1.19,-0.05,15.22,0.18,19.78,nan,nan,3.00,1.00,0.56,0.61,0.00,0.04,-0.10,1.10,-0.06,0.00,13.96,0.93,-10.31
XLP,-8.47,-6.91,-0.11,15.10,0.15,27.50,nan,nan,2.00,1.00,0.64,0.71,0.09,0.09,-0.15,1.06,-0.06,0.00,17.69,1.15,-14.76
MADE,12.98,10.52,0.05,14.08,0.25,32.79,nan,nan,7.00,4.00,0.53,0.58,0.28,0.14,0.15,1.17,0.02,100.00,13.49,1.05,-3.54
PKB,8.89,8.21,0.01,11.52,0.16,19.43,nan,nan,4.00,1.00,0.77,0.82,0.45,0.09,-0.09,1.13,0.02,97.62,16.43,1.32,-7.43
XLV,-5.71,6.38,0.09,2.48,-0.02,27.23,nan,nan,6.00,1.00,0.43,0.46,0.04,0.05,0.05,1.04,-0.06,23.81,14.85,-0.29,-6.67


In [ ]:
# Quick summary stat sentence for a given ticker

def print_summary_stat(df, ticker):

    if ticker not in df.index:
        print(f"Ticker {ticker} not found in DataFrame.")
        return
    row = df.loc[ticker]
    summary = []
    two_decimal_cols = [
        '% Days Above SMA200 6M',
        'MACD (ratio, %)',
        'MACD Histogram',
        'MACD Histogram Slope 20D',
        'Volatility Ratio 20D/100D',
        'OBV Slope 100D',
        'RS Slope 6M ex1M',
        'Ratio SMA200 Slope 50D',
        'Momentum Efficiency Slope',
        'RSI Slope 20D'
    ]  # 'RSI Up/Down Volatility Ratio' removed
    for col, val in row.items():
        # Remove anything in parentheses from the column name for display
        col_clean = col.split('(')[0].strip()
        # Format value
        if isinstance(val, float):
            if col in two_decimal_cols:
                val_str = f"{val:.2f}"
            else:
                val_str = f"{int(round(val))}"
        else:
            val_str = str(val)
        summary.append(f"{col_clean}: {val_str}")
    print(f"Summary for {ticker}: " + "; ".join(summary))


In [ ]:
# Curated filter for metrics_disp (displayed DataFrame)
# Flatten columns if MultiIndex (use only top level)
if isinstance(metrics_disp.columns, pd.MultiIndex):
    metrics_disp.columns = metrics_disp.columns.get_level_values(0)

# Define thresholds for each metric using the original (pre-rename) column names
criteria = {
    '12M Rel Strength Ex 1M': lambda x: -10 <= x <= 30,  # threshold applied
    '6M Rel Strength Ex 1M': lambda x: -10 <= x <= 30,
    '6M Rel Slope': lambda x: -0.1 <= x <= 0.3,
    '3M Rel Strength': lambda x: 0 <= x <= 35,
    '3M Rel Slope': lambda x: -0.03 <= x <= 0.4,
    'Ratio/SMA200': lambda x: -0.3 <= x <= 0.4,
    'Dist to SMA200 (%)': lambda x: -20 <= x <= 30,
    'SMA200 Slope (6M, 100x)': lambda x: -0.05 <= x <= 0.2,
    '%Days Ratio>SMA200 (6M)': lambda x: 20 <= x <= 100,
    'RS Volatility (6M)': lambda x: 0 <= x <= 37,
    'RS Volatility Slope (6M, 10000x)': lambda x: -0.3 <= x <= 0.3,
    'RS Max Drawdown (6M)': lambda x: -20 <= x <= 0,
    'Momentum Efficiency (6M)': lambda x: 0 <= x <= 0.2,
    'Momentum Efficiency Slope (6M, 100x)': lambda x: -0.1 <= x <= 0.25,
    'Distribution Days (30d)': lambda x: 0 <= x <= 7,
    'UVP (%) (40d)': lambda x: 0.4 <= x <= 0.7,
    'UVP Slope (40d, 1000x)': lambda x: -0.05 <= x <= 0.3,
    'PE Ratio': lambda x: 0 <= x <= 29,
    'FCF Margin': lambda x: 0.1 <= x <= 0.8,
    'ROIC': lambda x: 0.1 <= x <= 0.4,
    'Uptrend (Ratio>SMA200)': lambda x: x == 1 or x == 0,
}

# Apply filter: keep only rows that pass all non-None criteria and exist in the DataFrame
filtered = metrics_disp.copy()
for col, func in criteria.items():
    if func is not None and col in filtered.columns:
        filtered = filtered[filtered[col].apply(func)]
    elif func is not None and col not in filtered.columns:
        print(f"Warning: Column '{col}' not found in DataFrame and will be skipped.")

# Redisplay filtered DataFrame with same formatting
styled_filtered = filtered.style.format('{:.2f}').background_gradient(cmap='Greens')
styled_filtered

ticker


In [ ]:
filtered

""
ticker
